In [1]:
# Parameters
selected_fuel_type = 3


In [2]:
import json
import pandas as pd
import time
import re
import ast
import requests
import shutil
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support import expected_conditions as EC

In [3]:
# df view settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [4]:
CHROME_BINARY = shutil.which("chromium")
CHROMEDRIVER_PATH = shutil.which("chromedriver")

chrome_options = Options()
chrome_options.binary_location = CHROME_BINARY

chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-infobars")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1200,800")
chrome_options.add_argument("--blink-settings=imagesEnabled=false")

prefs = {
    "profile.managed_default_content_settings.images": 2,
    "profile.managed_default_content_settings.stylesheets": 2,
    "profile.managed_default_content_settings.fonts": 2,
    "profile.managed_default_content_settings.plugins": 2,
    "profile.managed_default_content_settings.notifications": 2,
}
chrome_options.add_experimental_option("prefs", prefs)

# Selenium 4 way to set capabilities:
chrome_options.set_capability("pageLoadStrategy", "eager")

service = Service(CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service, options=chrome_options)

In [5]:
# retrieving all the distinct car brands
u = "https://www.bilbasen.dk/brugt/bil?includeengroscvr=true&includeleasing=false"
data = json.loads(
    BeautifulSoup(requests.get(u, headers={"User-Agent": "Mozilla/5.0"}).text, "html.parser")
    .find("script", id="__NEXT_DATA__").string
)

c = []

def walk(x):
    if isinstance(x, list):
        labels = []
        for v in x:
            if isinstance(v, str):
                labels.append(v.strip())
            elif isinstance(v, dict):
                for k in ("label", "name", "title", "text", "value", "displayName"):
                    s = v.get(k)
                    if isinstance(s, str):
                        labels.append(s.strip())
                        break
        if len(labels) >= 30:
            uniq = sorted(set(labels))
            good = [
                s for s in uniq
                if s and len(s) <= 30 and s[0].isalpha() and s[0].isupper()
                and not any(ch.isdigit() for ch in s)
            ]
            if len(good) / len(uniq) > 0.8:
                c.append(uniq)
        for v in x:
            walk(v)
    elif isinstance(x, dict):
        for v in x.values():
            walk(v)

walk(data)

car_brands = sorted(c, key=len, reverse=True)[1]

In [6]:
fuel_options = {
    1: 'Benzin',
    2: 'Diesel',
    3: 'El',
    6: 'Hybrid - Benzin',
    8: 'Hybrid - Diesel',
    11: 'Plug-in Benzin',
    12: 'Plug-in Diesel'
}

In [7]:
def download_brand(brand: str, selected_fuel_type: str):
    page_listings = []
    base_url = (
        f"https://www.bilbasen.dk/brugt/bil/{brand}"
        f"?fuel={selected_fuel_type}&includeengroscvr=true&includeleasing=false"
    )
    driver.get(base_url)
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "span[data-e2e='pagination-total']"))
        )
    except TimeoutException:
        print(f"Timeout waiting for pagination on: {brand}")
        return None

    soup1 = BeautifulSoup(driver.page_source, "html.parser")
    page_tag = soup1.find('span', {'data-e2e': 'pagination-total'})
    if not (page_tag and page_tag.text.isdigit()):
        print(f"→ Skipping {brand}: 0 pages found")
        return None

    max_page = int(page_tag.text)

    for page in range(1, max_page + 1):
        paged_url = f"{base_url}&page={page}"
        print(f"Fetching {brand} page {page}/{max_page}")
        driver.get(paged_url)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        for art in soup.find_all("article"):
            if "".join(art.get("class", [])).startswith("Listing_listing"):
                for a in art.find_all("a", class_=lambda c: c and c.startswith("Listing_link")):
                    page_listings.append(a["href"])

    return page_listings

listings = []

for b in car_brands:
    pl = download_brand(b,str(selected_fuel_type))
    if pl:
        listings.extend(pl)

Timeout waiting for pagination on: AC


Fetching Abarth page 1/1


Fetching Aiways page 1/3


Fetching Aiways page 2/3


Fetching Aiways page 3/3


...

Fetching Xpeng page 2/3


Fetching Xpeng page 3/3


Timeout waiting for pagination on: Yugo


Fetching Zeekr page 1/1


Fetching firefly page 1/1


In [8]:
print(len(listings), len(set(listings)))

19913 19103


In [9]:
# Getting JSON data from each listing page (avoid navigating tag hierarchies). ~ 1 minute per 100 cars
all_parsed_data = []
total = len(set(listings))
last_report = time.time()

def func_wrapper_for_loop(i, link):
    global last_report

    # Progress monitoring after each 100 pages
    if i % 100 == 0 or i == total:
        now = time.time()
        elapsed = now - last_report
        mins, secs = divmod(int(elapsed), 60)
        print(
            f"{i}/{total} listings done "
            f"({i/total:.1%}) — last batch took {mins}m {secs}s",
            flush=True
        )
        last_report = now

    driver.get(link)
    car_soup = BeautifulSoup(driver.page_source, "html.parser")

    json_text = None
    for s in car_soup.find_all("script"):
        txt = (s.get_text() or "").lstrip()
        if txt.startswith("var _props"):
            m = re.search(r"var\s*_props\s*=\s*({.*?})\s*;", txt, flags=re.DOTALL)
            if m:
                json_text = m.group(1)
                break

    if not json_text:
        print("No _props JSON found on this page " + link)
        return

    try:
        parsed_data = json.loads(json_text)
        all_parsed_data.append(parsed_data)
    except Exception as e:
        print("Error parsing JSON:", e)

for i, link in enumerate(set(listings), start=1):
    func_wrapper_for_loop(i, link)


100/19103 listings done (0.5%) — last batch took 1m 43s


200/19103 listings done (1.0%) — last batch took 1m 48s


300/19103 listings done (1.6%) — last batch took 1m 46s


400/19103 listings done (2.1%) — last batch took 1m 47s


500/19103 listings done (2.6%) — last batch took 1m 47s


...

18800/19103 listings done (98.4%) — last batch took 1m 53s


18900/19103 listings done (98.9%) — last batch took 1m 48s


19000/19103 listings done (99.5%) — last batch took 1m 52s


19100/19103 listings done (100.0%) — last batch took 1m 54s


19103/19103 listings done (100.0%) — last batch took 0m 3s


In [10]:
# digest messy JSON data into a flat table of readable data
def extract_name_value(row):
    output = {}
    # Iterate over each cell in the row with its column label.
    for col, cell in row.items():
        # If the cell is a dictionary with the desired keys, transform it.
        if isinstance(cell, dict) and 'name' in cell and 'displayValue' in cell:
            output[cell['name']] = cell['displayValue']
        # If the cell is a string, try to parse it.
        elif isinstance(cell, str):
            try:
                d = ast.literal_eval(cell)
                if isinstance(d, dict) and 'name' in d and 'displayValue' in d:
                    output[d['name']] = d['displayValue']
                else:
                    # Not the desired structure, so keep the original cell under its column name.
                    output[col] = cell
            except Exception:
                # Parsing failed; keep the original cell.
                output[col] = cell
        else:
            # For any other type, simply keep the original cell.
            output[col] = cell
    return pd.Series(output)

In [11]:
all_listings = []

for entry in all_parsed_data:
    # try old key
    listing_data = entry.get("listing")

    # fall back to new path
    if listing_data is None:
        listing_data = []
        for q in (
            entry.get("props", {})
                 .get("pageProps", {})
                 .get("dehydratedState", {})
                 .get("queries", [])
        ):
            listing_data.extend(q.get("state", {}).get("data", {}).get("listings", []))

    if listing_data:
        # keep one level of nesting: 'vehicle.modelInformation' stays a dict
        flat = pd.json_normalize(listing_data, sep=".", max_level=1)
        all_listings.append(flat)

all_listings = pd.concat(all_listings, ignore_index=True)

In [12]:
all_listings = []

for entry in all_parsed_data:
    listing_data = entry.get('listing', {}) # access key values
    if listing_data:  # skip empty ones
        flattened = pd.json_normalize(listing_data) # flatten JSON data into flat table
        all_listings.append(flattened)

# Combine all the flattened listings into one DataFrame
all_listings = pd.concat(all_listings, ignore_index=True)

In [13]:
# Unpacking nested dictionaries into separate columns
df_model_info = all_listings['vehicle.modelInformation'].apply(pd.Series)
df_vehicle_details = all_listings['vehicle.details'].apply(pd.Series)
df_ratings = all_listings['vehicle.ratings.subRatings'].apply(pd.Series)
df_base = all_listings.drop(['vehicle.modelInformation', 'vehicle.details', 'vehicle.ratings.subRatings'], axis=1)
df_expanded = pd.concat([df_base, df_model_info, df_vehicle_details], axis=1)

In [14]:
rows = [extract_name_value(row) for _, row in df_expanded.iterrows()]
df_result = pd.DataFrame(rows)

<unknown>:1: SyntaxWarning: invalid decimal literal


<unknown>:1: SyntaxWarning: invalid decimal literal


<unknown>:1: SyntaxWarning: invalid decimal literal


In [15]:
benzin_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Geartype', 'Antal gear', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Brændstofforbrug','Cylindre', 'Airbags', 'Tankkapacitet','ABS-bremser', 'ESP', 'Periodisk afgift','CO2 udledning', 'Euronorm', 
        'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',
        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description'
]

el_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Energiforbrug', 'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC', 'Hurtig opladning DC', 'Opladningstid DC 10-80%',
        'Airbags', 'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift', 
        'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',
        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description'
    ]

In [16]:
columns = {
    'Benzin': benzin_cols,
    'Diesel': benzin_cols,
    'El':     el_cols,

}

In [17]:
today = pd.Timestamp.now().replace(microsecond=0)
yesterday = today - pd.Timedelta(days=1)
print(today, yesterday)
df_result.insert(0, 'scrape_timestamp', today)

2025-12-16 05:54:16 2025-12-15 05:54:16


In [18]:
df = df_result[columns[fuel_options[selected_fuel_type]]].copy()

In [19]:
today_str = today.strftime("%Y-%m-%d")

df.to_parquet(
    f"/home/pi-vault/projects/bilbasen_webscraping/data/{fuel_options[selected_fuel_type]}/"
    f"{fuel_options[selected_fuel_type]}_listings_{today_str}.parquet",
    index=True,
    engine="fastparquet",
)


In [20]:
driver.quit()